### Core Imports

In [ ]:
from pathlib import Path
import pandas as pd
import seaborn as sns
from pathlib import Path
import numpy as np
import shutil
from sklearn.model_selection import train_test_split
import seaborn as sns  # statistical plotting
sns.set_style('darkgrid')  # consistent dark grid background for plots
import matplotlib.pyplot as plt  # plotting primitives
from sklearn.model_selection import train_test_split  # dataset splitting helper
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)  # evaluation metrics

from PIL import Image  # image I/O and basic manipulation

import cv2  # OpenCV for image processing
import warnings  # control warnings
from tqdm import tqdm  # progress bars
tqdm.pandas()  # add progress bars to pandas operations

# Silence noisy warnings to keep notebook output clean
warnings.filterwarnings("ignore")
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import layers
from tensorflow.keras.applications import (
    EfficientNetB0,
    DenseNet121,
    VGG16,
    InceptionV3,
    MobileNetV2,
    ResNet50,
)
from pathlib import Path
import shutil
from sklearn.model_selection import train_test_split

import cv2  # OpenCV for image processing
import warnings  # control warnings
from tqdm import tqdm  # progress bars

# Silence noisy warnings to keep notebook output clean
warnings.filterwarnings("ignore")

print('modules loaded')  # quick sanity check that imports succeeded

sns.set_style("darkgrid")

### Raw Dataset Paths

In [ ]:
# Define base dataset path for the two datasets
dataset1_path = "dataset1"  # root folder holding train/val/test subfolders for dataset 1
dataset2_path = "dataset2"  # root folder holding train/val/test subfolders for dataset 2

### Resplitting Helper Functions

In [ ]:
def ensure_dir(path: Path):
    if not path.exists():
        path.mkdir(parents=True, exist_ok=True)


def resplit_dataset(source_root: Path, target_root: Path):
    if target_root.exists():
        shutil.rmtree(target_root)

    class_dirs = sorted(
        p.name for p in (source_root / "train").iterdir()
        if p.is_dir()
    )

    all_images = {cls: [] for cls in class_dirs}
    valid_exts = [".jpg", ".jpeg", ".png"]

    for split in ["train", "val", "test"]:
        for cls in class_dirs:
            split_cls_dir = source_root / split / cls
            if split_cls_dir.exists():
                all_images[cls].extend([
                    p for p in split_cls_dir.iterdir()
                    if p.is_file() and p.suffix.lower() in valid_exts
                ])

    for cls, imgs in all_images.items():
        if len(imgs) < 3:
            continue

        train_imgs, tmp_imgs = train_test_split(
            imgs, test_size=0.3, random_state=42, shuffle=True
        )

        val_imgs, test_imgs = train_test_split(
            tmp_imgs, test_size=2/3, random_state=42, shuffle=True
        )

        split_map = {
            "train": train_imgs,
            "val": val_imgs,
            "test": test_imgs
        }

        for split, split_imgs in split_map.items():
            for img_path in split_imgs:
                target_path = target_root / split / cls / img_path.name
                ensure_dir(target_path.parent)
                shutil.copy(img_path, target_path)

    return str(target_root)

### Execute Re-splitting

In [ ]:
source_root1 = Path(dataset1_path)
target_root1 = Path("dataset1_70_20_10")

source_root2 = Path(dataset2_path)
target_root2 = Path("dataset2_70_20_10")

dataset1_path = resplit_dataset(source_root1, target_root1)
dataset2_path = resplit_dataset(source_root2, target_root2)

print(dataset1_path)
print(dataset2_path)

### Verify Classes

In [ ]:
print("Dataset 1 classes:")
for cls in sorted((Path(dataset1_path) / "train").iterdir()):
    if cls.is_dir():
        print(cls.name)

print("\nDataset 2 classes:")
for cls in sorted((Path(dataset2_path) / "train").iterdir()):
    if cls.is_dir():
        print(cls.name)

### Count Images Per Class

In [ ]:
def count_images_per_class(split_path: Path):
    class_counts = {}
    for cls_dir in split_path.iterdir():
        if cls_dir.is_dir():
            class_counts[cls_dir.name] = len(list(cls_dir.glob("*")))
    return class_counts

### Split Counts

In [ ]:
train_counts1 = count_images_per_class(Path(dataset1_path) / "train")
train_counts2 = count_images_per_class(Path(dataset2_path) / "train")

val_counts1 = count_images_per_class(Path(dataset1_path) / "val")
val_counts2 = count_images_per_class(Path(dataset2_path) / "val")

test_counts1 = count_images_per_class(Path(dataset1_path) / "test")
test_counts2 = count_images_per_class(Path(dataset2_path) / "test")

print(train_counts1)
print(train_counts2)

### Visualize Training Class Distribution

In [ ]:
plt.figure(figsize=(16, 6))

plt.subplot(1, 2, 1)
plt.bar(train_counts1.keys(), train_counts1.values())
plt.title("Dataset 1 Train Distribution")
plt.xticks(rotation=45)

plt.subplot(1, 2, 2)
plt.bar(train_counts2.keys(), train_counts2.values())
plt.title("Dataset 2 Train Distribution")
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

### Build EDA DataFrame

In [ ]:
records = []

for split in ["train", "val", "test"]:
    for dataset_path in [dataset1_path, dataset2_path]:
        dataset_name = Path(dataset_path).name
        split_path = Path(dataset_path) / split

        for cls_dir in split_path.iterdir():
            if cls_dir.is_dir():
                for img_path in cls_dir.iterdir():
                    if img_path.is_file():
                        records.append({
                            "dataset": dataset_name,
                            "split": split,
                            "class": cls_dir.name,
                            "image_path": str(img_path)
                        })

eda_df = pd.DataFrame(records)
print("Total images:", len(eda_df))
eda_df.head()

### Pixel Range Check

In [ ]:

sample_image_path = eda_df["image_path"].iloc[0]

with Image.open(sample_image_path) as img:
    img_array = np.array(img)

print(img_array.shape)
print(img_array.min(), img_array.max())

### Sample Images Per Class

In [ ]:
samples_per_class = 3

for label in eda_df["class"].unique():
    sample_paths = eda_df[
        (eda_df["class"] == label) &
        (eda_df["split"] == "train")
    ]["image_path"].sample(samples_per_class, random_state=42)

    plt.figure(figsize=(12, 4))

    for i, path in enumerate(sample_paths):
        with Image.open(path) as img:
            plt.subplot(1, samples_per_class, i + 1)
            plt.imshow(img)
            plt.title(label)
            plt.axis("off")

    plt.show()

### Corruption Check

In [ ]:
broken = []

for path in eda_df["image_path"]:
    try:
        with Image.open(path) as img:
            img.verify()
    except Exception as e:
        broken.append((path, str(e)))

print("Broken images:", len(broken))

### Feature Extraction EDA

In [ ]:
def extract_features(image_path):
    try:
        with Image.open(image_path) as img:
            img = img.convert("RGB")
            img_array = np.array(img)

            width, height = img.size
            aspect_ratio = width / height

            mean_color = img_array.mean(axis=(0, 1))
            std_color = img_array.std(axis=(0, 1))

            gray = cv2.cvtColor(img_array, cv2.COLOR_RGB2GRAY)
            edges = cv2.Canny(gray, 100, 200)
            edge_density = edges.sum() / (width * height)

            return {
                "width": width,
                "height": height,
                "aspect_ratio": aspect_ratio,
                "mean_red": mean_color[0],
                "mean_green": mean_color[1],
                "mean_blue": mean_color[2],
                "std_red": std_color[0],
                "std_green": std_color[1],
                "std_blue": std_color[2],
                "edge_density": edge_density
            }
    except:
        return None


stat_df = eda_df["image_path"].progress_apply(extract_features).dropna().apply(pd.Series)
stat_df = pd.concat([eda_df, stat_df], axis=1)

stat_df.head()

### Declare train dataset

In [ ]:
TRAIN_DATASETS = {
    "dataset1": Path(dataset1_path),
    "dataset2": Path(dataset2_path),
}

### Config

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 20
LEARNING_RATE = 1e-4

MODEL_DIR = Path("models")
MODEL_DIR.mkdir(exist_ok=True)

TRAIN_DATASETS = {
    "dataset1": Path(dataset1_path),
    "dataset2": Path(dataset2_path),
}

### Data Generators

In [ ]:
def make_generators(dataset_root):
    train_gen = ImageDataGenerator(
        rescale=1.0 / 255,
        rotation_range=10,
        width_shift_range=0.05,
        height_shift_range=0.05,
        zoom_range=0.10,
        horizontal_flip=True,
        fill_mode="nearest",
    )

    plain_gen = ImageDataGenerator(rescale=1.0 / 255)

    train_ds = train_gen.flow_from_directory(
        dataset_root / "train",
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode="categorical",
        shuffle=True,
    )

    val_ds = plain_gen.flow_from_directory(
        dataset_root / "val",
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode="categorical",
        shuffle=False,
    )

    test_ds = plain_gen.flow_from_directory(
        dataset_root / "test",
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode="categorical",
        shuffle=False,
    )

    return train_ds, val_ds, test_ds

### # Inspect available class folders

In [ ]:
# Inspect available class folders under the training split for both datasets
print("\n Available classes in training splits:")
for cls in sorted((Path(dataset1_path) / "train").iterdir()):
    if cls.is_dir():
        print(f" for dataset 1")
        print(f"  {cls.name}")

for cls in sorted((Path(dataset2_path) / "train").iterdir()):
    if cls.is_dir():
        print(f" for dataset 2")
        print(f"  {cls.name}")

### Helper to count images per class inside

In [ ]:
# Helper to count images per class inside a split folders for both datasets for training, test and evaluation
def count_images_per_class(split_path: Path):
    class_counts = {}
    for cls_dir in split_path.iterdir():
        if cls_dir.is_dir():
            count = sum(1 for p in cls_dir.iterdir() if p.is_file())
            class_counts[cls_dir.name] = count
    return class_counts
        
# Count images per class for training, validation and test splits for both datasets
print("\n Image counts per class in training split:")
train_counts1 = count_images_per_class(Path(dataset1_path) / "train")
train_counts2 = count_images_per_class(Path(dataset2_path) / "train")
print(f" for dataset 1: {train_counts1}")
print(f" for dataset 2: {train_counts2}")
print("\n Image counts per class in validation split:")
val_counts1 = count_images_per_class(Path(dataset1_path) / "val")
val_counts2 = count_images_per_class(Path(dataset2_path) / "val")
print(f" for dataset 1: {val_counts1}")
print(f" for dataset 2: {val_counts2}")
print("\n Image counts per class in test split:")
test_counts1 = count_images_per_class(Path(dataset1_path) / "test")
test_counts2 = count_images_per_class(Path(dataset2_path) / "test")
print(f" for dataset 1: {test_counts1}")
print(f" for dataset 2: {test_counts2}")

### Visualize class distribution

In [ ]:
# Visualize class distribution in the training split for both datasets using bar plots
plt.figure(figsize=(16,6))

plt.subplot(1,3,1)
plt.bar(train_counts1.keys(), train_counts1.values())
plt.xlabel("Class") # x-axis label
plt.ylabel("Count") # y-axis label
plt.title("Dataset 1") # title

plt.subplot(1,3,2)
plt.bar(train_counts2.keys(), train_counts2.values())
plt.xlabel("Class") # x-axis label
plt.ylabel("Count") # y-axis label
plt.title("Dataset 2") # title

plt.show()

### Visualize class distribution

In [ ]:
def build_vit(input_shape, num_classes):
    patch_size = 16
    projection_dim = 64
    transformer_layers = 6
    num_heads = 4
    dropout_rate = 0.1
    num_patches = (input_shape[0] // patch_size) * (input_shape[1] // patch_size)

    def mlp(x, hidden_units):
        for units in hidden_units:
            x = layers.Dense(units, activation="gelu")(x)
            x = layers.Dropout(dropout_rate)(x)
        return x

    inputs = layers.Input(shape=input_shape)

    x = layers.Conv2D(projection_dim, kernel_size=patch_size, strides=patch_size)(inputs)
    x = layers.Reshape((num_patches, projection_dim))(x)

    positions = tf.range(start=0, limit=num_patches, delta=1)
    pos_embed = layers.Embedding(num_patches, projection_dim)(positions)
    x = x + pos_embed

    for _ in range(transformer_layers):
        x1 = layers.LayerNormalization()(x)
        attention = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=projection_dim
        )(x1, x1)

        x2 = layers.Add()([attention, x])

        x3 = layers.LayerNormalization()(x2)
        x3 = mlp(x3, [projection_dim * 2, projection_dim])

        x = layers.Add()([x3, x2])

    x = layers.GlobalAveragePooling1D()(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    return Model(inputs, outputs, name="VisionTransformer")

### ResNet50

In [ ]:
def build_resnet50(input_shape, num_classes):
    base = ResNet50(weights="imagenet", include_top=False, input_shape=input_shape)
    base.trainable = False

    x = GlobalAveragePooling2D()(base.output)
    x = Dense(256, activation="relu")(x)
    x = Dropout(0.3)(x)
    outputs = Dense(num_classes, activation="softmax")(x)

    return Model(base.input, outputs, name="ResNet50")

### EfficientNetB0

In [ ]:
def build_efficientnet(input_shape, num_classes):
    base = EfficientNetB0(weights="imagenet", include_top=False, input_shape=input_shape)
    base.trainable = False

    x = GlobalAveragePooling2D()(base.output)
    x = Dense(256, activation="relu")(x)
    outputs = Dense(num_classes, activation="softmax")(x)

    return Model(base.input, outputs, name="EfficientNetB0")

### MobileNetV2

In [ ]:
def build_mobilenet(input_shape, num_classes):
    base = MobileNetV2(weights="imagenet", include_top=False, input_shape=input_shape)
    base.trainable = False

    x = GlobalAveragePooling2D()(base.output)
    outputs = Dense(num_classes, activation="softmax")(x)

    return Model(base.input, outputs, name="MobileNetV2")

### InceptionV3

In [ ]:
def build_inception(input_shape, num_classes):
    base = InceptionV3(weights="imagenet", include_top=False, input_shape=input_shape)
    base.trainable = False

    x = GlobalAveragePooling2D()(base.output)
    outputs = Dense(num_classes, activation="softmax")(x)

    return Model(base.input, outputs, name="InceptionV3")

### VGG16

In [ ]:
def build_vgg16(input_shape, num_classes):
    base = VGG16(weights="imagenet", include_top=False, input_shape=input_shape)
    base.trainable = False

    x = GlobalAveragePooling2D()(base.output)
    outputs = Dense(num_classes, activation="softmax")(x)

    return Model(base.input, outputs, name="VGG16")

### DenseNet121

In [ ]:
def build_densenet(input_shape, num_classes):
    base = DenseNet121(weights="imagenet", include_top=False, input_shape=input_shape)
    base.trainable = False

    x = GlobalAveragePooling2D()(base.output)
    outputs = Dense(num_classes, activation="softmax")(x)

    return Model(base.input, outputs, name="DenseNet121")

### EfficientNetB0

In [ ]:
def build_efficientnet_variant(input_shape, num_classes):
    base = EfficientNetB0(weights="imagenet", include_top=False, input_shape=input_shape)
    base.trainable = False

    x = GlobalAveragePooling2D()(base.output)
    x = Dense(512, activation="relu")(x)
    x = Dropout(0.4)(x)
    outputs = Dense(num_classes, activation="softmax")(x)

    return Model(base.input, outputs, name="EfficientNetB0_Variant")

### Model Registry

In [ ]:
MODEL_BUILDERS = {
    "VisionTransformer": build_vit,
    "ResNet50": build_resnet50,
    "EfficientNetB0": build_efficientnet,
    "MobileNetV2": build_mobilenet,
    "InceptionV3": build_inception,
    "VGG16": build_vgg16,
    "DenseNet121": build_densenet,
    "EfficientNetB0_Variant": build_efficientnet_variant,
}

### Architecture-Centric Dual-Dataset Training

In [ ]:
results = []

for model_name, builder in MODEL_BUILDERS.items():
    print("\n" + "=" * 80)
    print(f"TRAINING ARCHITECTURE: {model_name}")
    print("=" * 80)

    model_histories = {}

    for dataset_name, dataset_root in TRAIN_DATASETS.items():
        print(f"\nTraining {model_name} on {dataset_name}")

        train_ds, val_ds, test_ds = make_generators(dataset_root)

        input_shape = IMG_SIZE + (3,)
        num_classes = len(train_ds.class_indices)

        model = builder(input_shape, num_classes)

        model.compile(
            optimizer=Adam(LEARNING_RATE),
            loss="categorical_crossentropy",
            metrics=["accuracy"],
        )

        history = model.fit(
            train_ds,
            validation_data=val_ds,
            epochs=EPOCHS,
            verbose=1,
        )

        test_loss, test_acc = model.evaluate(test_ds, verbose=0)

        #  Save using modern Keras format
        save_path = MODEL_DIR / f"{model_name}_{dataset_name}.keras"
        model.save(save_path)

        print(f" Saved model -> {save_path}")

        model_histories[dataset_name] = history.history

        results.append({
            "model": model_name,
            "dataset": dataset_name,
            "accuracy": float(test_acc),
            "loss": float(test_loss),
            "saved_path": str(save_path),
        })

    # ==================================================
    # Plot dataset1 vs dataset2 training history
    # ==================================================
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    # Accuracy
    for dataset_name, hist in model_histories.items():
        axes[0].plot(
            hist["accuracy"],
            marker="o",
            label=f"{dataset_name} train"
        )
        axes[0].plot(
            hist["val_accuracy"],
            linestyle="--",
            marker="x",
            label=f"{dataset_name} val"
        )

    axes[0].set_title(f"{model_name} Accuracy Comparison")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Accuracy")
    axes[0].legend()
    axes[0].grid(True)

    # Loss
    for dataset_name, hist in model_histories.items():
        axes[1].plot(
            hist["loss"],
            marker="o",
            label=f"{dataset_name} train"
        )
        axes[1].plot(
            hist["val_loss"],
            linestyle="--",
            marker="x",
            label=f"{dataset_name} val"
        )

    axes[1].set_title(f"{model_name} Loss Comparison")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Loss")
    axes[1].legend()
    axes[1].grid(True)

    plt.tight_layout()
    plt.show()

### Results Table

In [ ]:
results_df = pd.DataFrame(results)
results_df.sort_values(["model", "accuracy"], ascending=[True, False])

### Verify 16 Saved Models

In [ ]:
saved_models = list(MODEL_DIR.glob("*.keras"))

print(f" Total saved models: {len(saved_models)}")

for model_file in saved_models:
    print(model_file.name)